# AI Respondents Challenge — Starter Notebook

**Oxford LLMs 2026.** Given a survey respondent's answers to other questions, predict their
answer to each target question. This notebook is the minimal working pipeline:

1. load the challenge dataset,
2. pick the features (other survey answers) to show the model,
3. build a prompt per (respondent, target question) and call an open LLM on Nebius,
4. save a submission folder for upload.

It runs end-to-end as-is on a small demo slice. **It is a minimal working example, not a
recommended design.** How you predict is entirely up to you — the only fixed points are
the submission format and that any respondent information you use comes from the allowed
feature pool. Rules and schedule: see the challenge page.

In [1]:
%pip install -q datasets openai

In [20]:
import json
import math
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from sklearn.metrics import normalized_mutual_info_score
from tqdm.auto import tqdm

REPO = "oxford-llms/ai-respondents-challenge"

train    = load_dataset(REPO, "train",    split="train").to_pandas()
test     = load_dataset(REPO, "test",     split="train").to_pandas()
targets  = load_dataset(REPO, "targets",  split="train").to_pandas()
features = load_dataset(REPO, "features", split="train").to_pandas()

print(f"train {train.shape}, test {test.shape}, "
      f"{targets.question_id.nunique()} targets, {len(features)} allowed features")

train (5000, 345), test (1050, 280), 9 targets, 278 allowed features


## The targets and their label space

One row per answer option. **Your `prediction` strings must match a target's `label`
values exactly** — anything else scores zero. Options are listed in scale order.

In [3]:
for qid, g in targets.groupby("question_id", sort=False):
    print(f"{qid}: {g.question.iloc[0]}")
    print(f"     {' | '.join(g.label)}")

Q201: How often do you use daily newspaper to obtain information about this country and the world?
     Daily | Weekly | Monthly | Less than monthly | Never
Q73: How much confidence do you have in parliament?
     A great deal | Quite a lot | Not very much | None at all
Q227: In your view, how often are voters bribed in this country's elections?
     Very often | Fairly often | Not often | Not at all often
Q209: Have you ever signed a petition, might you consider signing one, or would you never do so under any circumstances?
     Have done | Might do | Would never do
Q33: Do you agree or disagree with the following statement: 'When jobs are scarce, men should have more right to a job than women'?
     Agree strongly | Agree | Neither agree nor disagree | Disagree | Disagree strongly
Q148: How worried are you about a civil war?
     Very much | A great deal | Not much | Not at all
Q17: Do you consider obedience to be an especially important quality for children to learn at home?
     Im

## Nebius API key

Open models run on the shared team allowance ([Nebius AI Studio](https://studio.nebius.com/)).
Get your team key from the organizers.

In [9]:
import os
from getpass import getpass
from openai import OpenAI

nebius_key = getpass("Nebius API key: ").strip()

if not nebius_key:
    raise ValueError("The Nebius API key is empty.")

# Save the key for the current notebook session
os.environ["NEBIUS_API_KEY"] = nebius_key

# Pass the same value directly to the client
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=nebius_key,
)

MODEL = "Qwen/Qwen3-32B"

print("Nebius client created successfully.")

Nebius API key: ··········
Nebius client created successfully.


Step 4 — Build metadata and robust code utilities

In [23]:
def canonical_code(value):
    """Convert numeric-looking survey codes into stable string keys."""
    if pd.isna(value):
        return None

    if isinstance(value, (np.integer, int)):
        return str(int(value))

    if isinstance(value, (np.floating, float)):
        number = float(value)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
        return format(number, "g")

    text = str(value).strip()

    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
        return format(number, "g")
    except (TypeError, ValueError):
        return text


def numeric_code(code):
    """Return a numeric representation when possible."""
    try:
        return float(code)
    except (TypeError, ValueError):
        return None


def sort_codes(codes):
    """Sort numeric codes numerically and other codes lexicographically."""
    return sorted(
        codes,
        key=lambda value: (
            numeric_code(value) is None,
            numeric_code(value) if numeric_code(value) is not None else value,
        ),
    )


def parse_values_json(value):
    """Parse feature response mappings safely."""
    if isinstance(value, dict):
        parsed = value
    elif pd.isna(value):
        parsed = {}
    else:
        parsed = json.loads(str(value))

    return {
        canonical_code(key): str(label)
        for key, label in parsed.items()
    }


qtext = dict(
    zip(
        features["variable"].astype(str),
        features["question"].astype(str),
    )
)

vmaps = {
    str(variable): parse_values_json(values_json)
    for variable, values_json in zip(
        features["variable"],
        features["values_json"],
    )
}

question_for = (
    targets
    .drop_duplicates("question_id")
    .set_index("question_id")["question"]
    .astype(str)
    .to_dict()
)

labels_for = (
    targets
    .groupby("question_id", sort=False)["label"]
    .apply(lambda values: [str(value) for value in values])
    .to_dict()
)

target_ids = list(labels_for)
target_id_set = set(target_ids)

allowed_features = [
    str(variable)
    for variable in features["variable"]
    if str(variable) in train.columns
]

# Safety check: target questions must not be used as respondent features.
candidate_features = [
    variable
    for variable in allowed_features
    if variable not in target_id_set
]

assert set(candidate_features).issubset(set(allowed_features))
assert not (set(candidate_features) & target_id_set)

print("Target IDs:", target_ids)
print("Candidate features:", len(candidate_features))

Target IDs: ['Q201', 'Q73', 'Q227', 'Q209', 'Q33', 'Q148', 'Q17', 'Q186', 'Q242']
Candidate features: 278


In [24]:
def build_target_code_maps(train, targets):
    """
    Build a validated mapping from raw training codes to official labels.

    Most targets already use the same codes in train and targets.option.
    Two challenge targets use coarsened output label spaces:

    Q186:
        raw 1–10 scale -> five two-point bins

    Q242:
        raw 0 is a special category;
        raw 1–10 -> five two-point bins
    """
    mappings = {}
    diagnostics = []

    for target_id, group in targets.groupby("question_id", sort=False):
        target_id = str(target_id)
        ordered_labels = group["label"].astype(str).tolist()

        observed_codes = sort_codes(
            set(
                train[target_id]
                .dropna()
                .map(canonical_code)
                .tolist()
            )
        )

        if not observed_codes:
            raise ValueError(f"No observed training codes for {target_id}.")

        # ------------------------------------------------------------
        # Special coarsening for Q186:
        # 1–2 Never, 3–4 Rarely, 5–6 Sometimes,
        # 7–8 Often, 9–10 Always.
        # ------------------------------------------------------------
        if target_id == "Q186":
            expected_labels = [
                "Never justifiable",
                "Rarely justifiable",
                "Sometimes justifiable",
                "Often justifiable",
                "Always justifiable",
            ]

            if ordered_labels != expected_labels:
                raise ValueError(
                    f"Unexpected Q186 labels: {ordered_labels}"
                )

            mapping = {
                "1": ordered_labels[0],
                "2": ordered_labels[0],
                "3": ordered_labels[1],
                "4": ordered_labels[1],
                "5": ordered_labels[2],
                "6": ordered_labels[2],
                "7": ordered_labels[3],
                "8": ordered_labels[3],
                "9": ordered_labels[4],
                "10": ordered_labels[4],
            }

            method = "Q186: raw 1–10 grouped into five two-point bins"

        # ------------------------------------------------------------
        # Special coarsening for Q242:
        # 0 is the special 'against democracy' category.
        # 1–2 Not essential, 3–4 Somewhat unimportant,
        # 5–6 Moderately essential, 7–8 Essential,
        # 9–10 Very essential.
        # ------------------------------------------------------------
        elif target_id == "Q242":
            expected_labels = [
                "It is against democracy",
                "Not essential",
                "Somewhat unimportant",
                "Moderately essential",
                "Essential",
                "Very essential",
            ]

            if ordered_labels != expected_labels:
                raise ValueError(
                    f"Unexpected Q242 labels: {ordered_labels}"
                )

            mapping = {
                "0": ordered_labels[0],
                "1": ordered_labels[1],
                "2": ordered_labels[1],
                "3": ordered_labels[2],
                "4": ordered_labels[2],
                "5": ordered_labels[3],
                "6": ordered_labels[3],
                "7": ordered_labels[4],
                "8": ordered_labels[4],
                "9": ordered_labels[5],
                "10": ordered_labels[5],
            }

            method = (
                "Q242: raw 0 kept as a special category; "
                "raw 1–10 grouped into five two-point bins"
            )

        # ------------------------------------------------------------
        # All other visible targets use direct code-to-label mappings.
        # ------------------------------------------------------------
        else:
            direct_mapping = {
                canonical_code(option): str(label)
                for option, label in zip(
                    group["option"],
                    group["label"],
                )
            }

            if not set(observed_codes).issubset(set(direct_mapping)):
                raise ValueError(
                    f"Direct mapping failed for {target_id}. "
                    f"Observed train codes: {observed_codes}. "
                    f"targets.option: {group['option'].tolist()}."
                )

            mapping = direct_mapping
            method = "direct targets.option mapping"

        missing_codes = set(observed_codes) - set(mapping)

        if missing_codes:
            raise ValueError(
                f"Target mapping for {target_id} is incomplete. "
                f"Missing codes: {sort_codes(missing_codes)}"
            )

        mappings[target_id] = mapping

        diagnostics.append({
            "question_id": target_id,
            "mapping_method": method,
            "observed_codes": observed_codes,
            "mapped_codes": sort_codes(mapping.keys()),
            "n_raw_codes": len(observed_codes),
            "n_output_labels": len(ordered_labels),
        })

    return mappings, pd.DataFrame(diagnostics)


option_to_label, mapping_diagnostics = build_target_code_maps(
    train=train,
    targets=targets,
)

display(mapping_diagnostics)

# This assertion prevents KeyError failures during validation.
for target_id in target_ids:
    observed = set(
        train[target_id]
        .dropna()
        .map(canonical_code)
        .tolist()
    )

    missing = observed - set(option_to_label[target_id])

    assert not missing, (
        f"{target_id} has unmapped codes: {sort_codes(missing)}"
    )

print("All observed target codes are mapped successfully.")

print("\nQ186 mapping:")
print(option_to_label["Q186"])

print("\nQ242 mapping:")
print(option_to_label["Q242"])

,question_id,mapping_method,observed_codes,mapped_codes,n_raw_codes,n_output_labels
0,Q201,direct targets.option mapping,"[1, 2, 3, 4, 5]","[1, 2, 3, 4, 5]",5,5
1,Q73,direct targets.option mapping,"[1, 2, 3, 4]","[1, 2, 3, 4]",4,4
2,Q227,direct targets.option mapping,"[1, 2, 3, 4]","[1, 2, 3, 4]",4,4
3,Q209,direct targets.option mapping,"[1, 2, 3]","[1, 2, 3]",3,3
4,Q33,direct targets.option mapping,"[1, 2, 3, 4, 5]","[1, 2, 3, 4, 5]",5,5
5,Q148,direct targets.option mapping,"[1, 2, 3, 4]","[1, 2, 3, 4]",4,4
6,Q17,direct targets.option mapping,"[1, 2]","[1, 2]",2,2
7,Q186,Q186: raw 1–10 grouped into five two-point bins,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",10,5
8,Q242,Q242: raw 0 kept as a special category; raw 1–...,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",11,6


All observed target codes are mapped successfully.

Q186 mapping:
{'1': 'Never justifiable', '2': 'Never justifiable', '3': 'Rarely justifiable', '4': 'Rarely justifiable', '5': 'Sometimes justifiable', '6': 'Sometimes justifiable', '7': 'Often justifiable', '8': 'Often justifiable', '9': 'Always justifiable', '10': 'Always justifiable'}

Q242 mapping:
{'0': 'It is against democracy', '1': 'Not essential', '2': 'Not essential', '3': 'Somewhat unimportant', '4': 'Somewhat unimportant', '5': 'Moderately essential', '6': 'Moderately essential', '7': 'Essential', '8': 'Essential', '9': 'Very essential', '10': 'Very essential'}


Step 5 — Select target-specific features using train only

In [25]:
MINIMUM_AVAILABILITY = 0.25
MINIMUM_PAIRED_ROWS = 100
TOP_K_FEATURES = 12


def rank_features_for_target(target_id):
    """Rank allowed features using only the labelled training data."""
    target_observed = train[target_id].notna()
    records = []

    for variable in candidate_features:
        feature_observed = train[variable].notna()
        paired_mask = target_observed & feature_observed

        paired_rows = int(paired_mask.sum())
        target_rows = int(target_observed.sum())

        if target_rows == 0:
            continue

        availability = paired_rows / target_rows

        if availability < MINIMUM_AVAILABILITY:
            continue

        if paired_rows < MINIMUM_PAIRED_ROWS:
            continue

        x = (
            train.loc[paired_mask, variable]
            .map(canonical_code)
            .astype(str)
        )
        y = (
            train.loc[paired_mask, target_id]
            .map(canonical_code)
            .astype(str)
        )

        if x.nunique() < 2 or y.nunique() < 2:
            continue

        nmi = float(
            normalized_mutual_info_score(y, x)
        )

        adjusted_score = nmi * math.sqrt(availability)

        records.append({
            "feature": variable,
            "question": qtext.get(variable, variable),
            "nmi": nmi,
            "availability": availability,
            "paired_rows": paired_rows,
            "adjusted_score": adjusted_score,
        })

    if not records:
        raise ValueError(
            f"No eligible features were found for target {target_id}."
        )

    return (
        pd.DataFrame(records)
        .sort_values(
            ["adjusted_score", "availability"],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )


feature_rankings = {}
CHOSEN_FEATURES = {}

for target_id in target_ids:
    ranking = rank_features_for_target(target_id)
    feature_rankings[target_id] = ranking

    selected = (
        ranking
        .head(TOP_K_FEATURES)["feature"]
        .tolist()
    )

    assert target_id not in selected
    assert set(selected).issubset(set(allowed_features))

    CHOSEN_FEATURES[target_id] = selected


for target_id in target_ids:
    print("\n" + "=" * 100)
    print(target_id, "—", question_for[target_id])

    display(
        feature_rankings[target_id][
            [
                "feature",
                "question",
                "adjusted_score",
                "nmi",
                "availability",
            ]
        ].head(TOP_K_FEATURES)
    )


Q201 — How often do you use daily newspaper to obtain information about this country and the world?


,feature,question,adjusted_score,nmi,availability
0,N_REGION_ISO,Which region do you currently live in?,0.128384,0.129791,0.978432
1,Q223,"If there were a national election tomorrow, fo...",0.077315,0.090572,0.728684
2,Q290,How do you identify when it comes to Racial be...,0.067538,0.071463,0.893167
3,Q203,How often do you use radio news to obtain info...,0.066378,0.066546,0.994961
4,Q267,In which country was your mother born?,0.058697,0.061030,0.925015
5,Q268,In which country was your father born? List of...,0.057768,0.060110,0.923604
6,Q266,In which country were you born?,0.056887,0.057522,0.978029
7,Q272,What language do you normally speak at home?,0.055223,0.055863,0.977222
8,Q202,How often do you use tv news to obtain informa...,0.054355,0.054388,0.998791
9,Q205,How often do you use email to obtain informati...,0.044217,0.044518,0.986495



Q73 — How much confidence do you have in parliament?


,feature,question,adjusted_score,nmi,availability
0,Q72,How much confidence do you have in political p...,0.369583,0.372576,0.984002
1,Q71,How much confidence do you have in the governm...,0.302403,0.306539,0.973198
2,Q74,How much confidence do you have in the civil s...,0.228967,0.230870,0.983586
3,Q76,How much confidence do you have in elections?,0.186349,0.189282,0.969250
4,Q70,How much confidence do you have in the courts?,0.164697,0.167397,0.968003
5,N_REGION_ISO,Which region do you currently live in?,0.124756,0.126166,0.977769
6,Q68,How much confidence do you have in labour unions?,0.117557,0.121712,0.932890
7,Q69,How much confidence do you have in the police?,0.116204,0.117480,0.978392
8,Q66,How much confidence do you have in the press?,0.114539,0.115394,0.985248
9,Q67,How much confidence do you have in television?,0.110754,0.111194,0.992105



Q227 — In your view, how often are voters bribed in this country's elections?


...

,feature,question,adjusted_score,nmi,availability
0,N_REGION_ISO,Which region do you currently live in?,0.057607,0.058219,0.979069
1,Q8,Do you consider independence to be an especial...,0.040650,0.040712,0.996952
2,Q290,How do you identify when it comes to Racial be...,0.039196,0.041513,0.891485
3,Q223,"If there were a national election tomorrow, fo...",0.038391,0.045048,0.726275
4,Q272,What language do you normally speak at home?,0.034785,0.035180,0.977647
5,Q266,In which country were you born?,0.033371,0.033754,0.977444
6,Q267,In which country was your mother born?,0.032820,0.034136,0.924406
7,Q268,In which country was your father born? List of...,0.032814,0.034156,0.922983
8,Q45,If society placed more emphasis on developing ...,0.019082,0.019359,0.971550
9,Q289CS9,Do you belong to a religion or religious denom...,0.017468,0.017604,0.984556



Q186 — How justifiable do you think sex before marriage is?


,feature,question,adjusted_score,nmi,availability
0,N_REGION_ISO,Which region do you currently live in?,0.212019,0.214423,0.977702
1,Q185,"Is divorce justifiable, in your view?",0.208582,0.209188,0.994211
2,Q193,How justifiable do you think it is to have cas...,0.200160,0.206336,0.941038
3,Q182,"Is homosexuality justifiable, in your view?",0.182148,0.186246,0.956475
4,Q184,"Is abortion justifiable, in your view?",0.154297,0.155181,0.988636
5,Q183,"Is prostitution justifiable, in your view?",0.144319,0.150233,0.922813
6,Q223,"If there were a national election tomorrow, fo...",0.143016,0.167627,0.727916
7,Q188,How justifiable do you think euthanasia is?,0.138343,0.140235,0.973199
8,Q187,How justifiable do you think suicide is?,0.130151,0.131068,0.986063
9,Q290,How do you identify when it comes to Racial be...,0.125287,0.132755,0.890652



Q242 — How essential is it for democracy that religious authorities interpret the laws? Please use a scale from 1 ('not at all essential') to 10 ('essential').


,feature,question,adjusted_score,nmi,availability
0,N_REGION_ISO,Which region do you currently live in?,0.202764,0.205049,0.977839
1,Q223,"If there were a national election tomorrow, fo...",0.125012,0.145305,0.740186
2,Q245,How essential do you think it is for a democra...,0.105883,0.109676,0.932039
3,Q243,How essential do you think it is for a democra...,0.097825,0.098136,0.993668
4,Q290,How do you identify when it comes to Racial be...,0.094911,0.100484,0.892149
5,Q241,How essential is it for democracy that governm...,0.089919,0.090282,0.991980
6,Q248,How essential do you think it is for a democra...,0.084285,0.084797,0.987970
7,Q272,What language do you normally speak at home?,0.078939,0.079854,0.977206
8,Q246,How essential do you think it is for a democra...,0.077950,0.078803,0.978472
9,Q247,How essential do you think it is for a democra...,0.074954,0.075457,0.986703


Step 6 — Compute target majority fallbacks using train only

In [26]:
def decode_target_value(target_id, value):
    """Convert one known training target code to its exact official label."""
    code = canonical_code(value)
    label = option_to_label[target_id].get(code)

    if label is None:
        raise ValueError(
            f"Unknown target code for {target_id}: {code}. "
            f"Available codes: {sort_codes(option_to_label[target_id].keys())}"
        )

    return label


majority_label_for = {}
label_prior_for = {}

for target_id in target_ids:
    observed_labels = (
        train[target_id]
        .dropna()
        .map(lambda value: decode_target_value(target_id, value))
    )

    shares = (
        observed_labels
        .value_counts(normalize=True)
        .reindex(labels_for[target_id], fill_value=0.0)
    )

    majority_label_for[target_id] = str(shares.idxmax())
    label_prior_for[target_id] = shares.to_dict()

majority_summary = pd.DataFrame([
    {
        "question_id": target_id,
        "majority_label": majority_label_for[target_id],
        "majority_share": label_prior_for[target_id][
            majority_label_for[target_id]
        ],
    }
    for target_id in target_ids
])

display(majority_summary)

,question_id,majority_label,majority_share
0,Q201,Never,0.404556
1,Q73,Not very much,0.360066
2,Q227,Fairly often,0.276425
3,Q209,Would never do,0.456907
4,Q33,Disagree,0.315535
5,Q148,Very much,0.360606
6,Q17,Not so important,0.671611
7,Q186,Never justifiable,0.402444
8,Q242,Not essential,0.357113


Step 7 — Build the strict prompt and robust parser

In [27]:
SYSTEM_PROMPT = """
You are a survey-response prediction model.

Infer how this specific real respondent answered the target survey
question. This is a classification task, not a request for your own
opinion.

Use only the respondent information supplied in the user message.
Prioritise directly related individual survey responses over broad
country-level assumptions.

Return exactly one label from the supplied answer list and no other text.
""".strip()


def answer_text(variable, value):
    """Decode one allowed feature response into readable text."""
    code = canonical_code(value)

    if code is None:
        return None

    return vmaps.get(variable, {}).get(code, code)


def build_prompt(respondent, target_id):
    """Build one prompt for one respondent-target pair."""
    profile_blocks = []

    for position, variable in enumerate(
        CHOSEN_FEATURES[target_id],
        start=1,
    ):
        value = respondent.get(variable)

        if pd.isna(value):
            continue

        decoded = answer_text(variable, value)

        profile_blocks.append(
            f"{position}. Question: {qtext[variable]}\n"
            f"   Answer: {decoded}"
        )

    profile = (
        "\n\n".join(profile_blocks)
        if profile_blocks
        else "No selected survey answers are available."
    )

    allowed_labels = "\n".join(
        f"- {label}"
        for label in labels_for[target_id]
    )

    country = respondent.get("country", "Not supplied")

    return f"""
Respondent country:
{country}

Related answers previously given by this respondent:

{profile}

Target question:

{question_for[target_id]}

Allowed answer labels:

{allowed_labels}

Predict the answer given by this specific respondent.

Output requirements:
- Return exactly one allowed label.
- Copy the label exactly as written.
- Do not provide an explanation.
- Do not include quotation marks.
- Do not output a numeric code.
- Do not add any other text.

Answer:
""".strip()


def normalise_reply(reply):
    """Apply conservative formatting normalisation."""
    cleaned = str(reply).strip()
    cleaned = cleaned.strip('"').strip("'")
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip()


def parse_label(reply, valid_labels):
    """Map a model reply back to one exact official label."""
    cleaned = normalise_reply(reply)

    # Exact match.
    for label in valid_labels:
        if cleaned == label:
            return label

    # Case-insensitive exact match.
    for label in valid_labels:
        if cleaned.casefold() == label.casefold():
            return label

    # Recover one unique valid label from a longer response.
    matches = [
        label
        for label in valid_labels
        if label.casefold() in cleaned.casefold()
    ]

    if len(matches) == 1:
        return matches[0]

    # When labels overlap, prefer the longest matching label.
    if len(matches) > 1:
        longest = sorted(matches, key=len, reverse=True)
        if len(longest[0]) > len(longest[1]):
            return longest[0]

    return None


def predict_raw(prompt):
    """Call the model with exponential-backoff retries."""
    last_error = None

    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                max_tokens=16,
                temperature=0,
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False,
                    }
                },
            )

            return response.choices[0].message.content

        except Exception as error:
            last_error = error

            if attempt < 4:
                time.sleep(min(2 ** attempt, 8))

    raise last_error

Step 8 — Test one prompt before running batches

In [28]:
EXAMPLE_TARGET = target_ids[0]

example_prompt = build_prompt(
    test.iloc[0],
    EXAMPLE_TARGET,
)

print(example_prompt)

raw_example_reply = predict_raw(example_prompt)
parsed_example_reply = parse_label(
    raw_example_reply,
    labels_for[EXAMPLE_TARGET],
)

print("\nRaw reply:", raw_example_reply)
print("Parsed reply:", parsed_example_reply)

assert parsed_example_reply in labels_for[EXAMPLE_TARGET]

Respondent country:
Andorra

Related answers previously given by this respondent:

1. Question: Which region do you currently live in?
   Answer: Escut d'Andorra la Vella

2. Question: If there were a national election tomorrow, for which party would you vote? If you do not know, which party appeals to you most?
   Answer: AND: Democrats for Andorra

3. Question: How do you identify when it comes to Racial belonging/ ethnic group in your country?
   Answer: AD: Caucasian white

4. Question: How often do you use radio news to obtain information about this country and the world?
   Answer: Monthly

5. Question: In which country was your mother born?
   Answer: Andorra

6. Question: In which country was your father born? List of codes in Annex
   Answer: Spain

7. Question: In which country were you born?
   Answer: Andorra

8. Question: What language do you normally speak at home?
   Answer: Catalan; Valencian

9. Question: How often do you use tv news to obtain information about this co

Step 9 — Quick labelled sanity check on train

In [29]:
VALIDATION_N_PER_TARGET = 10
WORKERS = 32


def known_target_label(respondent, target_id):
    """Read and decode one known training answer safely."""
    return decode_target_value(
        target_id=target_id,
        value=respondent[target_id],
    )


validation_jobs = []

for target_number, target_id in enumerate(target_ids):
    available = train[
        train[target_id].notna()
    ].copy()

    sample_size = min(
        VALIDATION_N_PER_TARGET,
        len(available),
    )

    sample = available.sample(
        n=sample_size,
        random_state=100 + target_number,
    )

    for _, respondent in sample.iterrows():
        validation_jobs.append((respondent, target_id))


def answer_validation(respondent, target_id):
    """Predict one known training target for a sanity check."""
    truth = known_target_label(
        respondent=respondent,
        target_id=target_id,
    )

    try:
        raw_reply = predict_raw(
            build_prompt(respondent, target_id)
        )

        parsed = parse_label(
            raw_reply,
            labels_for[target_id],
        )

        if parsed is None:
            prediction = majority_label_for[target_id]
            status = "parse_fallback"
        else:
            prediction = parsed
            status = "success"

    except Exception as error:
        prediction = majority_label_for[target_id]
        raw_reply = repr(error)
        status = "api_fallback"

    return {
        "respondent_id": respondent["respondent_id"],
        "question_id": target_id,
        "truth": truth,
        "prediction": prediction,
        "status": status,
        "raw_reply": raw_reply,
    }


validation_rows = []

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = [
        pool.submit(answer_validation, respondent, target_id)
        for respondent, target_id in validation_jobs
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Validation calls",
    ):
        validation_rows.append(future.result())


validation_predictions = pd.DataFrame(validation_rows)

validation_summary_rows = []

for target_id, group in validation_predictions.groupby(
    "question_id",
    sort=False,
):
    accuracy = float(
        (group["truth"] == group["prediction"]).mean()
    )

    majority_accuracy = float(
        (
            group["truth"]
            == majority_label_for[target_id]
        ).mean()
    )

    skill = (
        (accuracy - majority_accuracy)
        / (1.0 - majority_accuracy)
        if majority_accuracy < 1.0
        else 0.0
    )

    validation_summary_rows.append({
        "question_id": target_id,
        "accuracy": accuracy,
        "majority_accuracy": majority_accuracy,
        "approximate_skill": skill,
        "fallbacks": int(
            (group["status"] != "success").sum()
        ),
    })


validation_summary = pd.DataFrame(validation_summary_rows)

display(validation_summary)

print(
    "Mean approximate Skill:",
    validation_summary["approximate_skill"].mean(),
)

print("\nStatuses:")
display(validation_predictions["status"].value_counts())

Validation calls:   0%|          | 0/90 [00:00<?, ?it/s]

,question_id,accuracy,majority_accuracy,approximate_skill,fallbacks
0,Q201,0.7,0.6,0.250000,0
1,Q227,0.4,0.1,0.333333,0
2,Q73,0.9,0.0,0.900000,0
3,Q209,0.4,0.5,-0.200000,0
4,Q33,0.5,0.3,0.285714,0
5,Q148,0.5,0.3,0.285714,0
6,Q17,0.5,0.6,-0.250000,0
7,Q242,0.3,0.3,0.000000,0
8,Q186,0.6,0.5,0.200000,0


Mean approximate Skill: 0.20052910052910053

Statuses:


,count
status,
success,90


Step 10 — Run final test predictions

In [30]:
RUN_ID = "nmi12_strict_v2"
TEST_N = len(test)  # Use a smaller number only for a demo.
WORKERS = 32
CHECKPOINT_EVERY = 250

test_subset = (
    test
    if TEST_N == len(test)
    else test.sample(TEST_N, random_state=0)
).reset_index(drop=True)

checkpoint_path = Path(
    f"test_predictions_checkpoint_{RUN_ID}.csv"
)


def job_key(respondent_id, target_id):
    return f"{respondent_id}::{target_id}"


def answer_test(respondent, target_id):
    """Predict one hidden test target with deterministic fallbacks."""
    try:
        raw_reply = predict_raw(
            build_prompt(respondent, target_id)
        )

        parsed = parse_label(
            raw_reply,
            labels_for[target_id],
        )

        if parsed is None:
            prediction = majority_label_for[target_id]
            status = "parse_fallback"
        else:
            prediction = parsed
            status = "success"

    except Exception as error:
        prediction = majority_label_for[target_id]
        raw_reply = repr(error)
        status = "api_fallback"

    return {
        "job_key": job_key(
            respondent["respondent_id"],
            target_id,
        ),
        "respondent_id": respondent["respondent_id"],
        "question_id": target_id,
        "prediction": prediction,
        "status": status,
        "raw_reply": raw_reply,
    }


if checkpoint_path.exists():
    completed_df = pd.read_csv(checkpoint_path)
    completed_records = completed_df.to_dict("records")
    completed_keys = set(completed_df["job_key"].astype(str))
    print(f"Loaded {len(completed_records)} completed jobs.")
else:
    completed_records = []
    completed_keys = set()


pending_jobs = [
    (respondent, target_id)
    for _, respondent in test_subset.iterrows()
    for target_id in target_ids
    if job_key(
        respondent["respondent_id"],
        target_id,
    ) not in completed_keys
]

print("Pending API calls:", len(pending_jobs))

new_records = []

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    future_to_job = {
        pool.submit(
            answer_test,
            respondent,
            target_id,
        ): (
            respondent["respondent_id"],
            target_id,
        )
        for respondent, target_id in pending_jobs
    }

    for future in tqdm(
        as_completed(future_to_job),
        total=len(future_to_job),
        desc="Test calls",
    ):
        respondent_id, target_id = future_to_job[future]

        try:
            record = future.result()
        except Exception as error:
            # This is an additional last-resort guard.
            record = {
                "job_key": job_key(respondent_id, target_id),
                "respondent_id": respondent_id,
                "question_id": target_id,
                "prediction": majority_label_for[target_id],
                "status": "unexpected_fallback",
                "raw_reply": repr(error),
            }

        new_records.append(record)

        if len(new_records) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(
                completed_records + new_records
            ).to_csv(
                checkpoint_path,
                index=False,
            )


raw_predictions = pd.DataFrame(
    completed_records + new_records
).drop_duplicates(
    ["respondent_id", "question_id"],
    keep="last",
)

raw_predictions.to_csv(
    checkpoint_path,
    index=False,
)

predictions = raw_predictions[
    [
        "respondent_id",
        "question_id",
        "prediction",
    ]
].copy()

print("\nPrediction statuses:")
display(raw_predictions["status"].value_counts())

display(predictions.head(9))

Pending API calls: 9450


Test calls:   0%|          | 0/9450 [00:00<?, ?it/s]


Prediction statuses:


,count
status,
success,9450


,respondent_id,question_id,prediction
0,R20070701,Q186,Always justifiable
1,R20070701,Q17,Not so important
2,R20070701,Q73,A great deal
3,R20070701,Q227,Not at all often
4,R20070477,Q148,A great deal
5,R20070345,Q186,Sometimes justifiable
6,R20070345,Q201,Never
7,R20070345,Q33,Disagree
8,R20070345,Q209,Would never do


Step 11 — Validate the submission

In [30]:
expected_rows = len(test_subset) * len(target_ids)

assert len(predictions) == expected_rows, (
    f"Expected {expected_rows} rows, found {len(predictions)}."
)

assert not predictions.duplicated(
    ["respondent_id", "question_id"]
).any(), "Duplicate respondent-target pairs detected."

assert predictions["prediction"].notna().all(), (
    "Missing predictions detected."
)

assert predictions["respondent_id"].nunique() == len(test_subset)
assert predictions["question_id"].nunique() == len(target_ids)

for target_id, group in predictions.groupby(
    "question_id",
    sort=False,
):
    invalid = (
        set(group["prediction"])
        - set(labels_for[target_id])
    )

    assert not invalid, (
        f"Invalid labels for {target_id}: {invalid}"
    )

if TEST_N != len(test):
    print(
        "DEMO VALID, but this is not a full submission because "
        f"TEST_N={TEST_N} and len(test)={len(test)}."
    )
else:
    assert len(predictions) == 9450
    print("FULL SUBMISSION VALID")

print("Rows:", len(predictions))
print(
    "Fallbacks:",
    int((raw_predictions["status"] != "success").sum()),
)

Step 12 — Write the submission folder and ZIP

In [31]:
import shutil

sub = Path("submission")
shutil.rmtree(sub, ignore_errors=True)
(sub / "method").mkdir(parents=True, exist_ok=True)

predictions.to_csv(
    sub / "predictions.csv",
    index=False,
)

feature_disclosure = pd.DataFrame([
    {
        "question_id": target_id,
        "feature_variable_code": variable,
    }
    for target_id, selected_features in CHOSEN_FEATURES.items()
    for variable in selected_features
])

feature_disclosure.to_csv(
    sub / "features.csv",
    index=False,
)

example_respondent = test.iloc[0]

example_prompts = {
    target_id: build_prompt(
        example_respondent,
        target_id,
    )
    for target_id in target_ids
}

prompts_path = sub / "method" / "prompts.jsonl"

with prompts_path.open("w", encoding="utf-8") as file:
    for target_id in target_ids:
        record = {
            "question_id": target_id,
            "model": MODEL,
            "temperature": 0,
            "max_tokens": 16,
            "thinking_enabled": False,
            "selected_features": CHOSEN_FEATURES[target_id],
            "system_prompt": SYSTEM_PROMPT,
            "example_prompt": example_prompts[target_id],
        }

        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

method_text = f"""
# Method

We used {MODEL} with temperature 0 and model thinking disabled.

For each target question, we selected {TOP_K_FEATURES} target-specific
survey variables from the permitted feature pool. Feature ranking used
only the labelled training data. Candidate variables were ranked by
normalized mutual information with the target response, multiplied by
the square root of answer availability to mildly penalise sparse
features.

For every respondent-target pair, we constructed a separate zero-shot
prompt. Missing feature answers were omitted. The prompt included the
respondent's country, the selected question-answer pairs, the target
question, and the exact allowed answer labels.

The model was instructed to return exactly one label. Replies were
matched back to the official label set. When an API request failed or a
reply could not be parsed, we used the target-specific majority label
calculated from the training data.

Target code-to-label mappings were validated against all observed
training codes before prediction.
""".strip() + "\n"

(sub / "method" / "method.md").write_text(
    method_text,
    encoding="utf-8",
)

print("Submission files:")
for path in sub.rglob("*"):
    if path.is_file():
        print(" -", path)

Submission files:
 - submission/predictions.csv
 - submission/features.csv
 - submission/method/prompts.jsonl
 - submission/method/method.md


In [32]:
# Create the ZIP archive.
shutil.make_archive(
    "submission",
    "zip",
    root_dir=".",
    base_dir="submission",
)

print("Created submission.zip")

# Download in Google Colab.
from google.colab import files
files.download("submission.zip")

Created submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>